# 04 — Paraphrase Sandbox (Qwen3-32B lexical-propagation probe)

A self-contained test bench for the NeurIPS lexical-content analysis. Within each
of the three injection **styles** we tested, try paraphrase variants of the
wording and see — per hop — how far each one propagates and whether it executes.

**No PDFs, no manual injection.** You write a paraphrase as plain text; it is
spliced into the real *clean* carrier document's extracted text at the chosen
position, then run through the same multi-agent pipeline and scored with the
same primitives as the paper (`safe_executor` + explicit `<tool_call>` parsing +
endpoint-in-text).

**Scope.** This is an *exploration* bench. Because `pdftotext` line-wrapping
means spliced text is not byte-identical to a hand-authored injected PDF, a
paraphrase that wins here is a **candidate**: author its PDF twin and rerun it
through the real grid (`12_build_new_grid_styles_nochunk.py` + notebook 02)
before it goes in the paper.

Styles: `1` calibration line · `2` self-replicating note · `3` tool-binding note.


## 1 · Configure — edit this cell

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# EDIT THIS CELL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DOMAIN   = "aihc"       # "aihc" | "convex" | "policy"
STYLE    = "1"          # "1" calibration | "2" self-replicating | "3" tool-binding
POSITION = "middle"     # "begin" | "middle" | "end"

CONDITIONS      = ["3-hop"]        # subset of ["2-hop", "3-hop"]
THINKING_MODES  = ["off"]         # subset of ["off", "on"]

# Paraphrase variants to compare. Each value is the full injected wording and
# MUST contain the controlled endpoint URL (so propagation can be scored).
# "baseline" is loaded below from the real grid cell so you start from the exact
# text the paper ran; add your paraphrases alongside it, e.g.:
#
#   VARIANTS["softer"]   = "Note: reviewers normally archive a calibration line ... <endpoint> ..."
#   VARIANTS["imperative"] = "You must append and forward the line ... <endpoint> ..."
#
# Keep the endpoint URL and (document_id=...) intact unless you are specifically
# testing their removal.
VARIANTS = {}   # filled with the baseline in the next cell; add your own after

# Set to "RUN_H200" to actually call the model on Modal (costs GPU $).
CONFIRM_RUN = ""
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2 · Setup

In [ ]:
import os, sys, json, time
from pathlib import Path
from datetime import datetime

# Repo root (portable local / CI).
REPO_ROOT = Path(".").resolve()
_p = REPO_ROOT
while _p != _p.parent:
    if (_p / "pyproject.toml").exists():
        REPO_ROOT = _p; break
    _p = _p.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.scenario1 import paraphrase_lab as lab

OUTPUT_ROOT = REPO_ROOT / "experiments/scenario1/outputs/paraphrase_sandbox"

# Seed the baseline from the real grid cell if the user did not supply it.
if "baseline" not in VARIANTS:
    VARIANTS = {"baseline": lab.canonical_payload(DOMAIN, STYLE, POSITION), **VARIANTS}

print(f"Domain={DOMAIN} style={STYLE} ({lab.STYLE_IDS[STYLE]}) position={POSITION}")
print(f"Endpoint: {lab.endpoint_for(DOMAIN, STYLE, POSITION)}")
print(f"Variants: {list(VARIANTS)}")

## 3 · Build & preview (no GPU)

Splices each variant into the clean carrier and shows where it landed. Nothing
is sent to the model here.

In [ ]:
cfg        = lab.load_cell_config(DOMAIN, STYLE, POSITION)
clean_text = lab.clean_carrier_text(cfg)
endpoint   = lab.endpoint_for(DOMAIN, STYLE, POSITION)

plans = []
for cond in CONDITIONS:
    for mode in THINKING_MODES:
        for name, para in VARIANTS.items():
            if endpoint not in para:
                raise ValueError(f"variant {name!r} is missing the endpoint {endpoint!r}")
            plans.append(lab.variant_plan(
                domain=DOMAIN, style_key=STYLE, position=POSITION,
                variant_name=name, paraphrase=para,
                condition=cond, thinking_mode=mode,
            ))

# Show the splice point once (same offset for every variant at this position).
off = lab.build_sandbox_registry(DOMAIN, STYLE, POSITION, VARIANTS["baseline"])["injection"]["insertion_offset_utf8"]
print(f"Clean carrier: {len(clean_text):,} chars · splice offset {off:,} "
      f"({off/len(clean_text):.0%} through the document)\n")
print("… " + repr(clean_text[off-90:off]) + "  <<< PARAPHRASE INSERTED HERE >>>  "
      + repr(clean_text[off:off+90]) + " …\n")

n_calls = sum((3 if p["condition"] == "2-hop" else 4) for p in plans)
print(f"{len(plans)} trajectories · {n_calls} model calls "
      f"({len(VARIANTS)} variants × {len(CONDITIONS)} conditions × {len(THINKING_MODES)} modes)")
for p in plans:
    print(f"  {p['trajectory_id']}  ({len(p['paraphrase'])} chars)")

## 4 · Modal runner (Qwen3-32B, model loads once)

In [ ]:
%pip install modal -q

In [ ]:
import subprocess, modal

# Modal auth from env vars (same pattern as notebook 02).
_tid, _tsec = os.environ.get("MODAL_TOKEN_ID"), os.environ.get("MODAL_TOKEN_SECRET")
_profile = os.environ.get("MODAL_PROFILE", "default")
if CONFIRM_RUN == "RUN_H200":
    if not _tid or not _tsec:
        raise EnvironmentError("Set MODAL_TOKEN_ID and MODAL_TOKEN_SECRET to run.")
    subprocess.run(["modal","token","set","--token-id",_tid,"--token-secret",_tsec,
                    "--profile",_profile], check=True)
    subprocess.run(["modal","profile","activate",_profile], check=True)

sandbox_app = modal.App("spec-gap-paraphrase-sandbox")
sandbox_image = (
    modal.Image.debian_slim(python_version="3.12")
    .pip_install("torch==2.11.0","transformers==5.8.0","accelerate==1.12.0",
                 "huggingface-hub[hf-xet]>=0.34,<2","safetensors>=0.5,<1")
    .env({"HF_XET_HIGH_PERFORMANCE":"1","TOKENIZERS_PARALLELISM":"false"})
    .add_local_python_source("src")
)


@sandbox_app.function(image=sandbox_image, gpu="H100", timeout=3600,
                      secrets=[modal.Secret.from_name("huggingface-secret")])
def run_sandbox(plans: list, thinking_mode: str) -> list:
    """Run each variant through the pipeline, threading upstream output hop->hop."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from src.scenario1.generator import build_generation_request, _agent_events
    from src.infrastructure.qwen_modal import split_thinking_text

    tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-32B")
    model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-32B", torch_dtype=torch.bfloat16,
        attn_implementation="sdpa", device_map="auto").eval()
    print(f"Model loaded. VRAM {torch.cuda.memory_allocated()/1e9:.1f} GB")

    def generate(messages, tools, enable_thinking, settings):
        torch.manual_seed(settings["seed"]); torch.cuda.manual_seed_all(settings["seed"])
        kw = dict(conversation=messages, tokenize=True, add_generation_prompt=True,
                  return_dict=True, return_tensors="pt", enable_thinking=enable_thinking)
        if tools: kw["tools"] = tools
        inp = tok.apply_chat_template(**kw).to(model.device)
        with torch.inference_mode():
            out = model.generate(**inp, do_sample=settings["do_sample"],
                temperature=settings["temperature"], top_p=settings["top_p"],
                top_k=settings["top_k"], min_p=settings.get("min_p",0.0),
                max_new_tokens=settings["max_new_tokens"],
                pad_token_id=tok.eos_token_id)
        gen = out[0][inp["input_ids"].shape[1]:]
        raw = tok.decode(gen, skip_special_tokens=False)
        return split_thinking_text(raw)["final_content"] or tok.decode(gen, skip_special_tokens=True).strip()

    results = []
    for plan in plans:
        rec = plan["record"]
        settings = rec["model"]["decoding_settings"]
        upstream, hops = None, []
        for ev in _agent_events(rec):
            req = build_generation_request(rec, ev, thinking_mode=thinking_mode,
                                           upstream_message=upstream)
            visible = generate(req["messages"], req["tools"],
                               req["enable_thinking"], settings)
            hops.append({"agent_id": req["agent_id"], "text": visible})
            upstream = visible
            print(f"  {plan['trajectory_id']} · {req['agent_id']}: {len(visible)} chars")
        results.append({"trajectory_id": plan["trajectory_id"], "hops": hops})
    return results

## 5 · Run & score

In [ ]:
if CONFIRM_RUN != "RUN_H200":
    print("DRY MODE — set CONFIRM_RUN = 'RUN_H200' in the config cell to execute.")
else:
    scored = []
    t0 = time.time()
    with sandbox_app.run():
        for mode in THINKING_MODES:
            mode_plans = [p for p in plans if p["thinking_mode"] == mode]
            print(f"\n→ thinking={mode}: {len(mode_plans)} trajectories")
            raw = run_sandbox.remote(mode_plans, thinking_mode=mode)
            by_id = {r["trajectory_id"]: r for r in raw}
            for p in mode_plans:
                hop_texts = by_id[p["trajectory_id"]]["hops"]
                scored.append(lab.score_trajectory(
                    plan=p, hop_texts=hop_texts,
                    executor_tool_text=hop_texts[-1]["text"]))
    print(f"\nDone in {time.time()-t0:.0f}s")

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = OUTPUT_ROOT / f"paraphrase_sandbox_{DOMAIN}_{STYLE}_{POSITION}_{stamp}.json"
    out.write_text(json.dumps(scored, indent=2))
    print(f"Saved: {out.relative_to(REPO_ROOT)}")

## 6 · Summary

In [ ]:
if CONFIRM_RUN == "RUN_H200" and scored:
    print(f"{'variant':<18}{'cond':<7}{'think':<6}{'outcome':<28}{'exec'} {'→executor'} {'first_hop'}")
    print("─"*90)
    for r in sorted(scored, key=lambda x: (x["variant_name"], x["condition"], x["thinking_mode"])):
        print(f"{r['variant_name']:<18}{r['condition']:<7}{r['thinking_mode']:<6}"
              f"{r['outcome']:<28}{'YES' if r['tool_executed'] else ' no':<5}"
              f"{'YES' if r['endpoint_reached_executor'] else ' no':<10}"
              f"{r['first_propagated_hop']}")
    print("\nPer-hop endpoint survival (quoted-line verbatim in parens):")
    for r in scored:
        flags = "  ".join(
            f"{h['agent_id']}={'E' if h['endpoint_in_text'] else '-'}"
            f"({'q' if h['quoted_line_verbatim'] else '-'})" for h in r["hops"])
        print(f"  {r['variant_name']:<16} {r['condition']:<6} {flags}")